# Scrutor et minimal API : des endpoints par convention

**Série** Integrations-DotNet · **Notebook** Scrutor-1 · **Noyau** .NET (C#)

Ce notebook densifie l'axe « Scrutor / minimal API » de la veille *The Unexpected AI Stack* (Part 2, scaffolding du template `zeeq-tmpl`), différé tant que la surface d'accueil Aspire n'existait pas — elle est livrée depuis (série Aspire de ce dépôt).

## Objectifs

1. Comprendre le **scan d'assemblage par conventions** (Scrutor) appliqué à l'injection de dépendances.
2. Lire une convention d'**orchestration d'endpoints** (`IEndpoint` + `MapEndpoints`) où aucune route n'est énumérée à la main — code source réel, lu depuis le lab.
3. **Piloter un hôte minimal API réel** (`ScrutorLab/`, dossier frère de ce notebook) depuis le notebook : compilation, lancement sur port dynamique, lecture de la trace de découverte, appels HTTP, arrêt.
4. Mettre la mécanique en regard de son équivalent Python : routers FastAPI (`APIRouter` + `include_router`) et dépendances.

## Prérequis

- SDK .NET 9 (le notebook localise lui-même le runtime qui l'exécute).
- Aucune infrastructure externe : Kestrel local, pas de Docker, pas de WSL.

## Durée

Environ 40 minutes.

## Plan

1. Le scan DI par conventions — le mécanisme brut sur des services
2. La convention `IEndpoint` — lue dans le code source réel du lab
3. Le lab réel : un hôte minimal API piloté depuis le notebook
4. Parité FastAPI ↔ Scrutor, exercices, à retenir

**Traçabilité** : axe Scrutor/minimal API de #10473 (livrable : ligne de parité « orchestration d'endpoints par convention » remplie avec du code exécuté).

In [1]:
#r "nuget: Microsoft.Extensions.DependencyInjection, 9.0.3"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Extensions.DependencyInjection, 9.0.3

La version est épinglée une fois pour toutes les cellules (convention de la série). Ici, un seul package suffit : `Microsoft.Extensions.DependencyInjection` fournit `IServiceCollection`. Scrutor, le package tiers central du grain, vit dans le **lab compilé** (`ScrutorLab/ScrutorLab.csproj`, version 4.0.0 épinglée) : c'est lui qui l'exécute pour de vrai — on lit son code puis on pilote son hôte.

## 1. Le scan DI par conventions — le mécanisme brut

Enregistrer des services un par un (`AddTransient<IQuoteService, KellyQuote>()`...) fonctionne, mais chaque nouvelle implémentation exige une ligne dans la composition — et cette ligne vit loin de la classe qu'elle enregistre. Python résout le même problème par le **décorateur** (`@app.get` déclare la route là où elle est écrite) ; la contrepartie .NET est le **scan d'assemblage** : on décrit la *forme* des classes à enregistrer, le conteneur trouve toutes celles qui la satisfont.

In [2]:
using Microsoft.Extensions.DependencyInjection;

public interface IQuoteService
{
    string Name { get; }
    string Quote();
}

public sealed class KellyQuote : IQuoteService
{
    public string Name => "kelly";
    public string Quote() => "Mise proportionnelle a l'edge.";
}

public sealed class ShannonQuote : IQuoteService
{
    public string Name => "shannon";
    public string Quote() => "Diversifie, rebalance.";
}

var services = new ServiceCollection();
services.AddTransient<IQuoteService, KellyQuote>();     // une ligne par implementation
services.AddTransient<IQuoteService, ShannonQuote>();

using (var provider = services.BuildServiceProvider())
{
    Console.WriteLine($"Implementations enregistrees : {provider.GetServices<IQuoteService>().Count()}");
    foreach (var quote in provider.GetServices<IQuoteService>())
        Console.WriteLine($"  {quote.Name,-8} -> {quote.Quote()}");
}
Console.WriteLine($"Descripteurs poses a la main dans la composition : {services.Count}");

Implementations enregistrees : 2


  kelly    -> Mise proportionnelle a l'edge.


  shannon  -> Diversifie, rebalance.


Descripteurs poses a la main dans la composition : 2



warning CS1701: Assuming assembly reference 'System.ComponentModel, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'Microsoft.Extensions.DependencyInjection' matches identity 'System.ComponentModel, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.ComponentModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ComponentModel, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'Microsoft.Extensions.DependencyInjection.Abstractions' matches identity 'System.ComponentModel, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.ComponentModel', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.ComponentModel, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'Microsoft.Extensions.DependencyInjection.Abstractions' matches identity 'System.ComponentModel, Version=10.0.0.0, Cult

**Lecture du résultat.** L'enregistrement **fonctionne**, mais regardez le coût d'organisation : chaque implémentation a exigé sa propre ligne `AddTransient` dans la composition — une ligne qui vit loin de la classe qu'elle enregistre. Changer la durée de vie, ajouter une troisième implémentation : tout repasse par la composition.

Python résout le même problème par le **décorateur** (`@app.get` déclare la route là où elle est écrite). La contrepartie .NET est le **scan d'assemblage** : décrire la *forme* des classes à enregistrer — `FromAssemblyOf<IEndpoint>` + `AssignableTo<IEndpoint>()` — et laisser le conteneur les trouver.

**Pourquoi ce scan ne s'exécute-t-il pas dans ce notebook ?** Parce qu'un kernel interactif compile les cellules dans des assemblages de soumission dynamiques, que le scan par réflexion ne parcourt pas de façon fiable — la découverte par convention est une affaire d'application **compilée**. C'est précisément pour cela que le grain embarque le lab `ScrutorLab/` : le scan y tourne pour de vrai, et sa trace (`ENDPOINT::`, section 3) en donne la preuve runtime.

## 2. La convention `IEndpoint` — lue dans le code source réel du lab

Le dossier frère `ScrutorLab/` est un hôte minimal API complet. Sa convention tient en trois pièces : le contrat `IEndpoint`, le scan Scrutor qui découvre ses implémentations, le `MapEndpoints` qui les laisse poser leurs routes. On lit ces fichiers **tels qu'ils sont compilés** — le notebook n'en propose pas une copie de démonstration, et c'est ce code exact qui s'exécutera dans la section 3.

In [3]:
using System.IO;

var labDir = Path.Combine(Directory.GetCurrentDirectory(), "ScrutorLab");

void PrintSource(string relativePath)
{
    var fullPath = Path.Combine(labDir, relativePath);
    Console.WriteLine($"===== {relativePath} =====");
    var n = 0;
    foreach (var line in File.ReadLines(fullPath))
        Console.WriteLine($"{++n,3} | {line.Replace("\t", "    ")}");
    Console.WriteLine();
}

PrintSource("EndpointConvention.cs");

===== EndpointConvention.cs =====


  1 | // Convention d'orchestration des endpoints : contrat + scan DI.


  2 | // Ce fichier est le coeur du lab : il definit la convention que le notebook


  3 | // enseigne. Aucun endpoint n'est reference explicitement dans Program.cs :


  4 | // ils sont decouverts par scan d'assemblement (Scrutor).


...

 55 |             endpoint.Map(app);


 56 |         }


 57 |         return app;


 58 |     }


 59 | }


**Lecture du code.** Lignes 34-41 : le scan Scrutor transposé aux endpoints — `FromAssemblyOf<IEndpoint>` + `AssignableTo<IEndpoint>()`, avec une durée de vie paramétrable (`WithLifetime`). Lignes 48-58 : `MapEndpoints` résout toutes les implémentations enregistrées et **laisse chaque classe poser ses propres routes**. Le scope créé ligne 52 n'est pas décoratif : il permet à un endpoint enregistré `Scoped` de recevoir des dépendances à durée de vie raccourcie sans résolution racine.

À retenir du contrat (lignes 15-19) : une seule méthode, `Map(IEndpointRouteBuilder)` — l'endpoint apporte ses routes, le conteneur apporte ses dépendances.

In [4]:
PrintSource("Endpoints.cs");
PrintSource("Program.cs");

===== Endpoints.cs =====


  1 | // Endpoints du lab : chaque classe porte ses routes. Aucune n'est


  2 | // referencee dans Program.cs -- elles sont decouvertes par la convention.


  3 | 


  4 | using Microsoft.AspNetCore.Mvc;


...

 30 |     Console.WriteLine($"ENDPOINT::{endpoint.GetType().Name}");


 31 | 


 32 | app.MapEndpoints();                  // mapping automatique de tous les endpoints


 33 | 


 34 | app.Run();


**Lecture du code.** `Endpoints.cs` déclare quatre endpoints : `HealthEndpoint` (sans dépendance), `CatalogEndpoint` et `StatsEndpoint` (services métier **injectés par constructeur** — l'équivalent typé des `Depends` FastAPI), `EchoEndpoint` (liaison JSON du corps). Et le `Program.cs` complet tient en quelques lignes utiles dont **aucune ne nomme un endpoint** : services métier, `ScanEndpoints()`, trace de découverte, `MapEndpoints()`, `Run()`.

Écrire un endpoint de plus dans ce lab se réduit donc à : ajouter une classe dans `Endpoints.cs`. Reste à prouver que ça tient sur un vrai hôte.

## 3. Le lab réel : un hôte minimal API piloté depuis le notebook

Conformément au précédent de la série (notebook 09-Aspire-Harness), le notebook **pilote** le lab : compilation par le runtime du noyau, lancement sur un port dynamique (le lab imprime un marqueur `LISTENING::` sur sa sortie standard), lecture de la trace `ENDPOINT::` laissée par la découverte, appels HTTP, puis arrêt propre.

In [5]:
using System.Diagnostics;
using System.IO;

// Le MainModule du noyau est dotnet-interactive.exe (un shim), pas le SDK.
// On resout le vrai dotnet : DOTNET_ROOT (pose par l'environnement du noyau), puis le PATH.
var dotnetRoot = Environment.GetEnvironmentVariable("DOTNET_ROOT");
var dotnetCandidates = new List<string>();
if (!string.IsNullOrEmpty(dotnetRoot)) dotnetCandidates.Add(Path.Combine(dotnetRoot, "dotnet.exe"));
var where = Process.Start(new ProcessStartInfo("where.exe", "dotnet") { RedirectStandardOutput = true })!;
foreach (var line in (await where.StandardOutput.ReadToEndAsync()).Split(new[] { '\r', '\n' }, StringSplitOptions.RemoveEmptyEntries))
    dotnetCandidates.Add(line.Trim());
var dotnet = dotnetCandidates.FirstOrDefault(File.Exists)
    ?? throw new InvalidOperationException("dotnet.exe introuvable : definissez DOTNET_ROOT");
Console.WriteLine($"SDK dotnet retenu : {dotnet}");

var csproj = Path.Combine(labDir, "ScrutorLab.csproj");
var exe = Path.Combine(labDir, "bin", "Debug", "net9.0", "ScrutorLab.exe");
Console.WriteLine("Compilation du lab : ScrutorLab/ScrutorLab.csproj");

var buildStart = DateTime.UtcNow;
var build = Process.Start(new ProcessStartInfo(dotnet, $"build \"{csproj}\" --no-incremental --nologo -v q")
{
    RedirectStandardOutput = true,
    RedirectStandardError = true,
})!;
var buildOutput = await build.StandardOutput.ReadToEndAsync();
await build.WaitForExitAsync();
foreach (var line in buildOutput.Split('\n', StringSplitOptions.RemoveEmptyEntries).TakeLast(4))
    Console.WriteLine("  " + line.Trim());
Console.WriteLine($"build exit={build.ExitCode}");

if (build.ExitCode != 0 || !File.Exists(exe) || File.GetLastWriteTimeUtc(exe) < buildStart)
    throw new InvalidOperationException("le build n'a pas produit un exe frais -- voir sortie ci-dessus");
Console.WriteLine("exe frais produit (horodate apres le lancement du build)");

SDK dotnet retenu : C:\Users\jsboi\AppData\Local\Microsoft\dotnet\dotnet.exe


Compilation du lab : ScrutorLab/ScrutorLab.csproj


  0 Warning(s)


  0 Error(s)


  Time Elapsed 00:00:04.17


build exit=0


exe frais produit (horodate apres le lancement du build)


Le binaire qui exécute ce noyau est `dotnet-interactive.exe` — un *shim*, pas le SDK : lui demander un `build` afficherait son aide. Le notebook résout donc le vrai `dotnet` via `DOTNET_ROOT` (posé dans l'environnement du noyau), avec le `PATH` en repli, puis **vérifie que le build a produit un exécutable frais** — la garde échoue bruyamment si la compilation n'a pas eu lieu. Cette garde n'est pas décorative : elle a attrapé, pendant la rédaction de ce notebook, un build silencieusement sans effet.

In [6]:
var baseUrl = "";
var discovered = new List<string>();
var host = Process.Start(new ProcessStartInfo(exe)
{
    RedirectStandardOutput = true,
    RedirectStandardError = true,
    WorkingDirectory = labDir,
})!;

var deadline = DateTime.UtcNow + TimeSpan.FromSeconds(20);
while (DateTime.UtcNow < deadline)
{
    var line = await host.StandardOutput.ReadLineAsync();
    if (line is null) break;                              // process mort
    if (line.StartsWith("ENDPOINT::")) discovered.Add(line["ENDPOINT::".Length..]);
    if (line.StartsWith("LISTENING::")) { baseUrl = line["LISTENING::".Length..]; break; }
}

if (baseUrl.Length == 0)
    throw new InvalidOperationException("marqueur LISTENING absent -- le lab n'a pas demarre");
Console.WriteLine($"Lab demarre (pid {host.Id}), il ecoute sur {baseUrl}");
Console.WriteLine($"Trace de decouverte ({discovered.Count} endpoints) :");
foreach (var name in discovered)
    Console.WriteLine($"  {name}");

Lab demarre (pid 35016), il ecoute sur http://127.0.0.1:60849


Trace de decouverte (4 endpoints) :


  HealthEndpoint


  CatalogEndpoint


  EchoEndpoint


  StatsEndpoint


Port dynamique (`http://127.0.0.1:0` dans le lab) + marqueur `LISTENING::` : le notebook n'a **aucun port à convenir d'avance** avec le lab, donc pas de collision possible avec une autre exécution simultanée. Même discipline que `aspire run --isolated` vu dans la série Aspire, en version minimale.

Et la trace `ENDPOINT::` est la preuve **runtime** de la découverte : ces quatre noms viennent de la résolution DI du scan — aucun n'a été écrit dans `Program.cs`.

In [7]:
using System.Net.Http;

using (var http = new HttpClient { BaseAddress = new Uri(baseUrl) })
{
    Console.WriteLine("GET  /health  -> " + await http.GetStringAsync("/health"));
    Console.WriteLine("GET  /catalog -> " + await http.GetStringAsync("/catalog"));

    var payload = new StringContent("{\"message\": \"scrutor\"}", System.Text.Encoding.UTF8, "application/json");
    var echo = await http.PostAsync("/echo", payload);
    Console.WriteLine("POST /echo    -> " + await echo.Content.ReadAsStringAsync());

    Console.WriteLine("GET  /stats   -> " + await http.GetStringAsync("/stats"));
    Console.WriteLine("GET  /stats   -> " + await http.GetStringAsync("/stats"));
}

GET  /health  -> {"status":"ok","endpoint":"HealthEndpoint"}


GET  /catalog -> {"count":3,"models":["qwen2.5-vl-7b","llama-3.1-8b-instruct","mistral-nemo-12b"],"endpoint":"CatalogEndpoint"}


POST /echo    -> {"received":"scrutor","chars":7}


GET  /stats   -> {"requests":1,"endpoint":"StatsEndpoint"}


GET  /stats   -> {"requests":2,"endpoint":"StatsEndpoint"}


**Lecture du résultat.** Quatre endpoints ont répondu, et aucun n'apparaît dans le `Program.cs` du lab — ils ont été découverts par la convention. Trois détails à relever :

- **`/catalog` et `/stats` portent un état** : `CatalogEndpoint` et `StatsEndpoint` reçoivent leurs services (`ICatalogService`, `IStatsService`) **par constructeur** — c'est la DI qui les résout au moment du scan, exactement comme un `Depends` FastAPI, mais vérifié à la compilation.
- **`/stats` passe de 1 à 2** : le compteur vit dans un singleton `IStatsService` ; deux appels, deux incréments. La durée de vie est un choix d'orchestration, pas un accident.
- **`/echo` fait l'aller-retour** : liaison JSON du corps vers `EchoRequest` par le minimal API, sans configuration.

In [8]:
// taskkill, appele depuis ce processus, emet sa sortie en UTF-8 :
// on declare l'encodage cote lecture pour eviter tout mojibake.
var kill = Process.Start(new ProcessStartInfo("taskkill", $"/PID {host.Id} /T /F")
{
    RedirectStandardOutput = true,
    StandardOutputEncoding = System.Text.Encoding.UTF8,
})!;
Console.WriteLine((await kill.StandardOutput.ReadToEndAsync()).Trim());
await kill.WaitForExitAsync();
await host.WaitForExitAsync();
Console.WriteLine($"Lab arrete : processus termine={host.HasExited}");

Opération réussie : le processus de PID 35016 (processus enfant de PID 10024) a été
arrêté.


Lab arrete : processus termine=True


Le nettoyage suit le précédent de la série (notebook 03-Aspire-Observabilite) : `taskkill /PID <pid> /T /F` termine l'arborescence — l'exécutable du lab et tout processus enfant. Un notebook qui démarre un hôte doit le rendre : c'est ce que fait cette cellule, et c'est pourquoi le notebook est rejouable à volonté.

## 4. Parité avec Python — routers FastAPI contre convention Scrutor

La ligne de parité visée par la série (EPIC #10473) : **orchestration d'endpoints par convention**.

| Aspect | FastAPI (Python) | Scrutor + `IEndpoint` (.NET) |
|---|---|---|
| Déclarer une route | décorateur `@router.get(...)` sur une fonction | classe implémentant `IEndpoint`, routes posées dans `Map` |
| Regrouper | `APIRouter` puis `include_router` explicite | scan d'assemblage — aucune inscription manuelle |
| Dépendances | `Depends(...)` par handler | constructeur de la classe d'endpoint, résolu par la DI |
| Durée de vie | implicite (fonction par requête) | explicite et choisie (`Transient` / `Scoped` / `Singleton`) |
| Vérification | au chargement (introspection runtime) | à la compilation (types, signatures) |

Le rapprochement est structurel : là où FastAPI **décore** puis **agrège** (`include_router`), .NET **type** puis **découvre** (`IEndpoint` + scan). La différence de fond tient au moment où l'erreur se voit : route mal orthographiée ou dépendance manquante se paient au chargement côté Python, à la compilation côté .NET.

## Exercices

Les trois exercices modifient le **lab** puis rejouent les cellules de compilation et de lancement de la section 3 — la trace `ENDPOINT::` et les appels HTTP servent de vérification.

In [9]:
// Exercice 1 : endpoint horloge
// TODO Etudiant : ajoutez a ScrutorLab/Endpoints.cs la classe
//
//   public sealed class TimeEndpoint : IEndpoint
//   {
//       public void Map(IEndpointRouteBuilder app)
//           => app.MapGet("/time", () => DateTime.UtcNow.ToString("o"));
//   }
//
// Etape 1 : re-executez la cellule de compilation (section 3).
// Etape 2 : re-executez la cellule de lancement : ENDPOINT::TimeEndpoint doit apparaitre.
// Etape 3 : ajoutez un appel GET /time dans la cellule HTTP et re-executez-la.
var traceAttendue = "Exercice a completer";  // TODO Etudiant : quelles lignes la trace contiendra-t-elle en plus ?
Console.WriteLine($"Exercice a completer : {traceAttendue}");

Exercice a completer : Exercice a completer


In [10]:
// Exercice 2 : duree de vie des endpoints
// TODO Etudiant : recopiez ScanEndpoints dans ScrutorLab/EndpointConvention.cs sous le nom
// ScanEndpointsScoped, avec WithLifetime(ServiceLifetime.Scoped), branchez-le dans Program.cs,
// puis justifiez en une phrase quand Scoped bat Transient pour un endpoint qui consomme un DbContext.
var justification = "Exercice a completer";  // TODO Etudiant : votre justification ici
Console.WriteLine($"Exercice a completer : {justification}");

Exercice a completer : Exercice a completer


In [11]:
// Exercice 3 : exclusion selective au scan
// TODO Etudiant : modifiez le predicat AddClasses de ScanEndpoints (ScrutorLab/EndpointConvention.cs)
// pour EXCLURE HealthEndpoint :
//   classes.AssignableTo<IEndpoint>().Where(t => t != typeof(HealthEndpoint))
// Recompilez, relancez : la trace ENDPOINT:: ne doit plus citer HealthEndpoint.
var prediction = "Exercice a completer";  // TODO Etudiant : GET /health renverra-t-il 200 ou 404 ? Pourquoi ?
Console.WriteLine($"Exercice a completer : prediction = {prediction}");

Exercice a completer : prediction = Exercice a completer


## À retenir

- **Scrutor scanne, la convention orchestre** : `FromAssemblyOf` + `AssignableTo<IEndpoint>` découvre les endpoints ; `MapEndpoints` les laisse poser leurs routes. Ajouter un endpoint = écrire une classe, rien d'autre.
- **Les dépendances traversent le constructeur** de la classe d'endpoint, résolues par la DI au moment du scan — l'équivalent typé des `Depends` FastAPI.
- **Un notebook peut piloter un vrai hôte** : compilation par le runtime du noyau, port dynamique lu sur stdout, trace de découverte, appels HTTP, `taskkill` final. Aucune infrastructure exigée.

## Vérifiez votre compréhension

1. Pourquoi `MapEndpoints` crée-t-il un scope alors que le mapping n'a lieu qu'une fois au démarrage ?
2. Que se passerait-il si deux implémentations de `IEndpoint` posaient le même gabarit de route ?
3. Où réside la différence de *moment du diagnostic* entre un `include_router` oublié (FastAPI) et un endpoint hors assemblage de scan (Scrutor) ?

## Pour aller plus loin

- `CopilotSDK/CopilotAgent.App/Program.cs` dans ce dépôt : la même convention en production sur un agent Copilot (dont un endpoint SSE).
- La série Aspire du même dossier : l'orchestration monte d'un cran (multi-processus, dashboard, télémétrie) — la convention `IEndpoint` y est la brique locale, l'AppHost la brique système.

***

**Traçabilité** : axe « Scrutor/minimal API » (Part 2 *The Unexpected AI Stack*), axe 2 différé par la distillation, livré ici — `See #22358`, Epic #10473.